<a href="https://colab.research.google.com/github/manish190502/Agentic_AI_Learning/blob/main/devops_rag_assistant.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [33]:
runbooks_corpus = [
    # Document 1 (PostgreSQL)
    """INCIDENT-001: PostgreSQL Connection Pool Exhaustion
Severity: High
Symptom: Application logs show "OperationalError: FATAL: remaining connection slots are reserved for non-replication superuser connections".
Resolution: SSH into the database proxy and run: sudo systemctl restart pgbouncer
Rollback: Do NOT restart the primary PostgreSQL container directly.""",

    # Document 2 (Kubernetes)
    """INCIDENT-002: Kubernetes Pod CrashLoopBackOff
Severity: Medium
Symptom: Deployment pods are repeatedly crashing and restarting continuously.
Resolution: Check previous logs using: kubectl logs <pod-name> --previous
Rollback: Do not force-delete the namespace, as this tears down service meshes.""",

    # Document 3 (Redis OOM)
    """INCIDENT-003: Redis Out of Memory (OOM) Panic
Severity: High
Symptom: Redis cache rejects writes with "OOM command not allowed when used memory > 'maxmemory'".
Resolution: Connect to Redis CLI and flush volatile keys or adjust maxmemory policy: redis-cli config set maxmemory-policy volatile-lru
Rollback: Avoid a hard service kill on the primary cache node to prevent total cache thrashing.""",

    # Document 4 (High CPU Load)
    """INCIDENT-004: Linux Host High CPU Utilization
Severity: High
Symptom: System load average exceeds 90% and SSH responses are sluggish.
Resolution: Identify the top CPU-consuming process using: top -b -n 1 | head -n 20, then gracefully terminate the runaway process if necessary: kill -15 <PID>
Rollback: Never run a blind `kill -9` on system processes or database daemons without checking parent IDs.""",

    # Document 5 (Nginx 502 Bad Gateway)
    """INCIDENT-005: Nginx Upstream 502 Bad Gateway
Severity: Medium
Symptom: Users receive HTTP 502 errors when hitting the frontend load balancer.
Resolution: Check Nginx error logs at /var/log/nginx/error.log and restart the upstream application daemon: sudo systemctl restart gunicorn
Rollback: Validate nginx configuration syntax with `nginx -t` before restarting the service to prevent complete traffic drop."""
]

doc_ids = ["incident_pg", "incident_k8s", "incident_redis", "incident_cpu", "incident_nginx"]

In [34]:
chroma_client = chromadb.Client()

In [35]:
collection = chroma_client.get_or_create_collection(name="new_collection")


In [36]:
collection.add(documents=runbooks_corpus,
               ids=doc_ids)

In [37]:
user_query = "Cache is rejecting writes because maxmemory is reached, what command fixes it?"
results = collection.query(query_texts=[user_query],n_results=1)
retrieved_chunk = results["documents"][0][0]
print(retrieved_chunk)

INCIDENT-003: Redis Out of Memory (OOM) Panic
Severity: High
Symptom: Redis cache rejects writes with "OOM command not allowed when used memory > 'maxmemory'".
Resolution: Connect to Redis CLI and flush volatile keys or adjust maxmemory policy: redis-cli config set maxmemory-policy volatile-lru
Rollback: Avoid a hard service kill on the primary cache node to prevent total cache thrashing.


In [38]:
from pydantic import BaseModel,Field
class Incident_response(BaseModel):
  alert_title:str = Field(description="Name of the production incident.")
  risk_level: str = Field(description="Severity level (e.g., High, Medium, Low)")
  recommended_command: str = Field(
      description="The exact terminal command to run."
  )
  rollback_steps: str = Field(description="Steps to undo the action safely.")
  is_grounded: bool = Field(
      description=(
          "True if the answer is derived strictly from context, False otherwise."
      )
  )

In [39]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
tokenizer = AutoTokenizer.from_pretrained("google/flan-t5-base")
model = AutoModelForSeq2SeqLM.from_pretrained("google/flan-t5-base")

Loading weights:   0%|          | 0/282 [00:00<?, ?it/s]

[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


In [47]:
augmented_prompt = f"""You are a precise Devops incident assistant. You will answer only the provided context
Context:{retrieved_chunk}
Question:{user_query}
Resolution Command:
"""

In [48]:
inputs = tokenizer(augmented_prompt,return_tensors="pt",max_length=512,truncation=True)
outputs = model.generate(**inputs,max_length=512,do_sample=False)

In [49]:
generated_answer = tokenizer.decode(outputs[0],skip_special_tokens=True)
print(generated_answer)

connect to Redis CLI and flush volatile keys or adjust maxmemory policy: redis-cli config set maxmemory-policy volatile-lru


In [43]:
# 1. Instantiate your Pydantic object with the results of your pipeline
final_incident = Incident_response(
    alert_title="PostgreSQL Connection Pool Exhaustion",
    risk_level="High",
    recommended_command=generated_answer,
    rollback_steps=(
        "Do NOT restart primary PostgreSQL directly; use PgBouncer."
    ),
    is_grounded=True,  # (You can wire your judge verdict here!)
)

# 2. Print the clean, type-safe schema
print("\n--- Final Pydantic Enforced Output ---")
print(final_incident)
print(f"\n[+] Verified Command to Execute: {final_incident.recommended_command}")


--- Final Pydantic Enforced Output ---
alert_title='PostgreSQL Connection Pool Exhaustion' risk_level='High' recommended_command='rollback' rollback_steps='Do NOT restart primary PostgreSQL directly; use PgBouncer.' is_grounded=True

[+] Verified Command to Execute: rollback
